## Imports

In [1]:
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama
import os
from dotenv import load_dotenv
import requests
from global_variables import LOCAL_MODEL_NAME, LOCAL_BASE_URL


## Globale Variablen

In [2]:
system_prompt = "Du bist ein Tutor für Vorlesungsinhalte. Beantworte Fragen nur auf Basis des bereitgestellten Kontexts. Erkläre klar, korrekt und verständlich. Wenn Informationen fehlen oder unsicher sind, sage das ausdrücklich. Erfinde nichts und spekuliere nicht. Nutze Fachbegriffe korrekt und erkläre sie kurz, wenn nötig."

LOCAL = True

## Tools

Retreival Tool

In [3]:
# Das Tool stellt Anfragen an die VectorDB und bekommt die entsprechenden Chunks zurück
@tool('search_lecture_docs', description='Retrieves information from Lecture related Documents')
def search_lecture_docs(query: str):
    response = requests.post(
    "http://127.0.0.1:8000/query",
    json={
        "query": query,
        "n": 3
    })
    return response.json()

## Initialisierungen

In [4]:
# Model für Lokale Ollama Modelle
model_local = ChatOllama(
    base_url=LOCAL_BASE_URL,
    model=LOCAL_MODEL_NAME,
)
# Model für Nvidia NIM API
load_dotenv()
model_api = ChatOpenAI(
    model="meta/llama-3.1-8b-instruct",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"],
)

agent = create_agent(model_local if LOCAL else model_api, tools=[search_lecture_docs], system_prompt=system_prompt)

## Ingest (PDF → Markdown → DB)

In [7]:
import sys
from pathlib import Path

# server/ auf den Importpfad legen (dort liegen db.py und pdf_to_markdown.py)
SERVER_DIR = Path.cwd() / "server"
if str(SERVER_DIR) not in sys.path:
    sys.path.insert(0, str(SERVER_DIR))

import server.db as db # oeffnet die ChromaDB (server/VectorDB)
from server.pdf_to_markdown import convert_all, PROCESSED_DIR


def ingest_all(convert: bool = True, reset: bool = True):
    """Kompletter Ingest: PDFs -> Markdown -> ChromaDB.

    1. convert=True: wandelt alle PDFs aus server/data/raw/ nach Markdown um.
    2. schreibt alle Markdown-Dateien aus server/data/processed/ in die Vektor-DB.

    reset=True leert die Collection vorher, damit ein erneuter Aufruf keine
    Duplikate erzeugt (add_document nutzt zufaellige UUIDs).

    Hinweis: Der DB-Server (db.py) sollte dabei NICHT laufen – ChromaDB oeffnet
    die lokale DB nur aus einem Prozess. Reihenfolge: erst ingest_all() im
    Notebook, dann den Server starten und ueber das Notebook Fragen stellen.
    """
    if reset:
        try:
            db.client.delete_collection("VectorDB")
        except Exception:
            pass
        db.collection = db.client.get_or_create_collection(name="VectorDB")

    if convert:
        convert_all()

    md_files = sorted(PROCESSED_DIR.glob("*.md"))
    for md_path in md_files:
        db.add_document(str(md_path))
        print(f"in DB geladen: {md_path.name}")

    print(f"\nFertig: {len(md_files)} Markdown-Datei(en), "
          f"Collection enthaelt jetzt {db.collection.count()} Chunks.")

In [8]:
# Einmalig ausfuehren, um die DB zu befuellen (Server dabei NICHT laufen lassen):
ingest_all()

[1/2] 08_LinAbb.pdf


[INFO] 2026-08-21 15:20:50,586 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-21 15:20:50,632 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\aergh\Desktop\studium\Master\NLP\rag-lecture-tutor\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-21 15:20:50,633 [RapidOCR] main.py:63: Using C:\Users\aergh\Desktop\studium\Master\NLP\rag-lecture-tutor\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-21 15:20:50,821 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-21 15:20:50,826 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\aergh\Desktop\studium\Master\NLP\rag-lecture-tutor\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-21 15:20:50,827 [RapidOCR] main.py:63: Using C:\Users\aergh\Desktop\studium\Master\NLP\rag-lecture-tutor\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 202

08_LinAbb.pdf -> 08_LinAbb.md (6262 Zeichen)
[2/2] Is Semantic Chunking Worth the Computational Cost.pdf
Is Semantic Chunking Worth the Computational Cost.pdf -> Is Semantic Chunking Worth the Computational Cost.md (71855 Zeichen)


C:\Users\aergh\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:15<00:00, 5.31MiB/s]


in DB geladen: 08_LinAbb.md
in DB geladen: example.md
in DB geladen: Is Semantic Chunking Worth the Computational Cost.md

Fertig: 3 Markdown-Datei(en), Collection enthaelt jetzt 1189 Chunks.


In [117]:
prompt = str(input())
result = agent.invoke({"messages": [("user", prompt)]})
print(result["messages"][-1].content)

Ja – die Organisation unterstützt die fachliche Weiterbildung der Mitarbeitenden.  
Jeder Mitarbeiter kann **jährlich bis zu 1 000 €** für berufliche Weiterbildungsmaßnahmen beantragen. Damit steht eine finanzielle Förderung für Kurse, Seminare, Zertifizierungen oder ähnliche Qualifizierungsangebote zur Verfügung.
